In [ ]:
data_product_identifier = "mdm_extract_dat1"
source_system_id = "mdm"

In [ ]:
import lakehouse_toolbelt
import datetime
import pyspark.sql.functions as F

In [ ]:
data_product = lakehouse_toolbelt.DataProduct("mdm_extract_dat1")

In [ ]:
## Can be used to select only named datasets for testing & development
filtered_datasets = {dataset_name: dataset for dataset_name, dataset in data_product.datasets.items() if dataset_name not in ['']}

for dataset_name, dataset in filtered_datasets.items():

    print(f"Starting to process dataset: {dataset_name}")

    latest_time_from_change_history = dataset.get_delta_change_history_dpmeta_loaded_to_dataplatform()
    
    try:  
        df_raw = (spark.read.format("csv")
            .option("recursiveFileLookup", "true")
            .option("header", True)
            .option("sep", ";")
            )

        if latest_time_from_change_history:
            print(f"Adding modifiedAfter -filter for timestamp: {latest_time_from_change_history}")
            df_raw.option("modifiedAfter", latest_time_from_change_history) 

        df_raw=df_raw.load(dataset.file_path_raw)
    
        #Add input file name and source system information
        df_raw_with_input_columns = (df_raw.withColumn("dpmeta_input_file_name", F.input_file_name())
                                    .withColumn("dpmeta_source_system_id", F.lit(source_system_id)) 
                                    )

        if dataset.timestamp_columns_to_convert:
            df_raw_with_input_columns = lakehouse_toolbelt.convert_timestamps(df_raw_with_input_columns, dataset.timestamp_columns_to_convert)

        #Add metadata columns
        df_raw_with_dpmeta_columns = lakehouse_toolbelt.add_dpmeta_columns(dataframe=df_raw_with_input_columns, dataset=dataset)
            
        #Drop potential duplicates for same entity version
        df_raw_with_dpmeta_columns = df_raw_with_dpmeta_columns.dropDuplicates(["dpmeta_uid", "dpmeta_object_version_start_time"])

        #Validate df schema before upsert
        df_validated = lakehouse_toolbelt.align_dataframe_to_delta_change_history_schema(df_raw_with_dpmeta_columns, dataset)

        #Perform upsert
        lakehouse_toolbelt.upsert_delta_change_history(source_dataframe = df_validated, dataset = dataset, debug_return =False)

    # pyspark will fail to infer the schema if no data is found and throws an error.
    except Exception as caught_exception:
        if "[UNABLE_TO_INFER_SCHEMA]" in str(caught_exception):
            print("Filtered incoming dataset is empty. Skipping upsert.")
        else:
            raise Exception(caught_exception)